# 🤖 Introduzione all'AI Agentica
## Workflow Design Patterns — 5 esempi in Python con Ollama + LangChain

---

Questo notebook illustra i **5 design pattern fondamentali dei workflow AI**.
A differenza degli *agenti* (che decidono autonomamente cosa fare), i **workflow** seguono
un percorso di esecuzione predefinito e deterministico.

| # | Pattern | Descrizione breve |
|---|---------|-------------------|
| 1 | **Prompt Chaining** | Output di un LLM → input del successivo |
| 2 | **Parallelization** | Più chiamate LLM eseguite in parallelo |
| 3 | **Routing** | Smistamento dell'input verso handler specializzati |
| 4 | **Orchestrator-Subagents** | Un orchestratore coordina worker specializzati |
| 5 | **Evaluator-Optimizer** | Un valutatore migliora iterativamente l'output |

> **Requisiti:** Ollama in esecuzione su `localhost:11434` con il modello `llama3.2` scaricato.

---
## ⚙️ Setup — Installazione e configurazione

[Opzionale] Crea il tuo virtual env con <b>python -m venv agenticintro .<b>

In [1]:
# Installa le dipendenze necessarie
%pip install -q langchain langchain-ollama langchain-core

Note: you may need to restart the kernel to use updated packages.


In [3]:
# ── Imports globali ──────────────────────────────────────────────────────────
from typing import Any


import asyncio
import textwrap
from concurrent.futures import ThreadPoolExecutor, as_completed

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# ── Configurazione del modello ───────────────────────────────────────────────
MODEL = "qwen3.5:9b"          # Cambia con il modello che hai scaricato su Ollama
BASE_URL = "http://localhost:11434"

llm: Any = ChatOllama(
    model=MODEL,
    base_url=BASE_URL,
    temperature=0.7,
)

# Helper per stampare in modo leggibile
def stampa(titolo: str, testo: str):
    separatore = "─" * 60
    print(f"\n{separatore}")
    print(f"📌 {titolo}")
    print(separatore)
    print(textwrap.fill(testo.strip(), width=70))
    print()

# Verifica connessione a Ollama
test = llm.invoke([HumanMessage(content="Rispondi solo: OK")])
print(f"✅ Ollama risponde correttamente — modello: {MODEL}")

✅ Ollama risponde correttamente — modello: qwen3.5:9b


---
## 1️⃣ Pattern: Prompt Chaining

### Cos'è?
Ogni step trasforma l'output del precedente. Il flusso è **lineare e sequenziale**:

```
Input → [LLM 1] → testo_1 → [LLM 2] → testo_2 → [LLM 3] → Output finale
```

### Quando usarlo?
Quando il task si decompone in **fasi ordinate** dove ogni fase dipende dalla precedente.

### Esempio
Pipeline per produrre un articolo di blog:  
`Idea → Scaletta → Bozza → Versione finale migliorata`

In [4]:
# ── 1. PROMPT CHAINING ────────────────────────────────────────────────────────
# Obiettivo: scrivere un post di blog in 3 step concatenati.

parser = StrOutputParser()

# Step 1 — Genera la scaletta
prompt_scaletta = ChatPromptTemplate.from_messages([
    ("system", "Sei un esperto di content marketing. Rispondi sempre in italiano."),
    ("human", "Crea una scaletta in 4 punti per un articolo su: {argomento}"),
])

# Step 2 — Scrivi la bozza
prompt_bozza = ChatPromptTemplate.from_messages([
    ("system", "Sei un copywriter esperto. Rispondi sempre in italiano."),
    ("human", "Scrivi una bozza di articolo di blog partendo da questa scaletta:\n\n{scaletta}"),
])

# Step 3 — Migliora il tono
prompt_finale = ChatPromptTemplate.from_messages([
    ("system", "Sei un editor editoriale. Rispondi sempre in italiano."),
    ("human", 
     "Revisiona questo articolo: rendi il tono più coinvolgente e aggiungi una call-to-action finale.\n\n{bozza}"),
])

# Composizione della chain con il pipe operator (|)
chain_scaletta = prompt_scaletta | llm | parser
chain_bozza    = prompt_bozza    | llm | parser
chain_finale   = prompt_finale   | llm | parser

# Esecuzione sequenziale — ogni output diventa input del passo successivo
argomento = "intelligenza artificiale nella vita quotidiana"

scaletta = chain_scaletta.invoke({"argomento": argomento})
stampa("STEP 1 — Scaletta", scaletta)

bozza = chain_bozza.invoke({"scaletta": scaletta})
stampa("STEP 2 — Bozza", bozza)

articolo_finale = chain_finale.invoke({"bozza": bozza})
stampa("STEP 3 — Articolo finale", articolo_finale)


────────────────────────────────────────────────────────────
📌 STEP 1 — Scaletta
────────────────────────────────────────────────────────────
Ecco una scaletta ottimizzata per un articolo di content marketing,
strutturata per catturare l'attenzione, fornire valore e generare
engagement.  **Titolo suggerito:** *Intelligenza Artificiale nella
Vita Quotidiana: Invisibile ma Indispensabile*  **1. Il punto di
ingresso (Hook): "L'IA è già qui, senza che te ne accorga"** *
**Obiettivo:** Rompere il mito che l'IA sia una tecnologia riservata
solo a laboratori informatici o aziende giganti. *   **Contenuto
chiave:** Elenca esempi subliminali dell'IA nel quotidiano (es:
algoritmi di raccomandazione su Netflix/Spotify, filtri delle foto sul
telefono, navigazione GPS che evita traffico, assistenti vocali). *
*Hook da usare:* "Probabilmente stai leggendo questo articolo perché
l'algoritmo sa che ti interessa."  **2. Il cuore del valore
(Benefits): "Come l'IA sta semplificando la tua routine"** *
*

---
## 2️⃣ Pattern: Parallelization

### Cos'è?
Più chiamate LLM vengono eseguite **in contemporanea** su input differenti, poi i risultati vengono **aggregati**:

```
           ┌─► [LLM A] ─► risultato_A ─┐
Input ─────┼─► [LLM B] ─► risultato_B ─┼─► Aggregazione → Output
           └─► [LLM C] ─► risultato_C ─┘
```

### Quando usarlo?
- Task **indipendenti** che possono girare in parallelo (risparmio di tempo)
- **Voting / consensus**: stessa domanda a più modelli o temperature diverse
- Elaborazione di **documenti multipli** in batch

### Esempio
Analisi di un prodotto da 3 prospettive simultanee: marketing, tecnica e UX.

In [5]:
# ── 2. PARALLELIZATION ────────────────────────────────────────────────────────
# Obiettivo: analizzare un prodotto da 3 prospettive in parallelo,
# poi sintetizzare i risultati.

prodotto = "un assistente AI vocale per anziani"

# Definiamo 3 task indipendenti: (nome, system_prompt, domanda)
tasks = [
    (
        "Marketing",
        "Sei un esperto di marketing. Rispondi in italiano in 3-4 frasi.",
        f"Quali sono i principali punti di forza di mercato per '{prodotto}'?",
    ),
    (
        "Tecnica",
        "Sei un ingegnere software senior. Rispondi in italiano in 3-4 frasi.",
        f"Quali sono le principali sfide tecniche nello sviluppare '{prodotto}'?",
    ),
    (
        "UX/Design",
        "Sei un UX designer esperto. Rispondi in italiano in 3-4 frasi.",
        f"Quali principi UX sono fondamentali per progettare '{prodotto}'?",
    ),
]

def esegui_task(nome: str, system: str, domanda: str) -> tuple[str, str]:
    """Esegue una singola chiamata LLM e restituisce (nome, risposta)."""
    risposta = llm.invoke([
        SystemMessage(content=system),
        HumanMessage(content=domanda),
    ])
    return nome, risposta.content

# Esecuzione parallela con ThreadPoolExecutor
risultati = {}
with ThreadPoolExecutor(max_workers=3) as executor:
    futures = {
        executor.submit(esegui_task, nome, system, domanda): nome
        for nome, system, domanda in tasks
    }
    for future in as_completed(futures):    # Non appena un thread completa vaso na scrivere il risultato
        nome, risposta = future.result()
        risultati[nome] = risposta
        print(f"  ✓ Completato: {nome}")

# Stampa i risultati parziali
for prospettiva, testo in risultati.items():
    stampa(f"Prospettiva: {prospettiva}", testo)

# Aggregazione: un LLM sintetizza le 3 analisi
sintesi_input = "\n\n".join(
    f"=== {k} ===\n{v}" for k, v in risultati.items()
)
sintesi = llm.invoke([
    SystemMessage(content="Sei un consulente strategico. Rispondi in italiano."),
    HumanMessage(content=(
        f"Hai ricevuto 3 analisi indipendenti sul prodotto '{prodotto}'.\n"
        f"Sintetizzale in un paragrafo conclusivo che integri tutti i punti di vista:\n\n"
        f"{sintesi_input}"
    )),
])
stampa("SINTESI FINALE", sintesi.content)

  ✓ Completato: UX/Design
  ✓ Completato: Marketing
  ✓ Completato: Tecnica

────────────────────────────────────────────────────────────
📌 Prospettiva: UX/Design
────────────────────────────────────────────────────────────
Fondamentale è garantire un'interazione vocale estremamente chiara e
ad alta fedeltà, adattata a eventuali deficit uditivi o cognitivi
degli utenti senior. Bisogna adottare un tono empatico e rassicurante,
fornendo feedback immediati per confermare le azioni compiute dal
sistema e costruire fiducia. Infine, l'esperienza deve essere priva di
distrazioni, privilegiando comandi naturali che riducano il carico
cognitivo e favoriscano l'autonomia quotidiana in sicurezza.


────────────────────────────────────────────────────────────
📌 Prospettiva: Marketing
────────────────────────────────────────────────────────────
Il principale punto di forza risiede nel soddisfare le esigenze di una
fascia demografica in rapida crescita tramite un'interfaccia intuitiva
che supera le 

---
## 3️⃣ Pattern: Routing

### Cos'è?
Un **classificatore** (LLM o regola) esamina l'input e lo **smista** verso il handler più appropriato:

```
                        ┌─► Handler A (tecnico)
Input → [Classifier] ──┼─► Handler B (commerciale)
                        └─► Handler C (supporto)
```

### Quando usarlo?
Quando diversi tipi di input richiedono **trattamenti specializzati** differenti.
Evita di usare un unico prompt gigante per tutto.

### Esempio
Un sistema di ticketing che classifica le email dei clienti e le gestisce con prompt dedicati.

In [6]:
# ── 3. ROUTING ────────────────────────────────────────────────────────────────
# Obiettivo: classificare un'email cliente e rispondere con il prompt giusto.

# Handlers specializzati per ogni categoria
HANDLERS = {
    "tecnico": (
        "Sei un tecnico esperto di supporto IT. "
        "Rispondi in italiano con precisione tecnica e passi numerati."
    ),
    "rimborso": (
        "Sei un addetto al servizio clienti empatico. "
        "Rispondi in italiano con tono professionale e comprensivo, "
        "spiegando la procedura di rimborso."
    ),
    "commerciale": (
        "Sei un consulente commerciale entusiasta. "
        "Rispondi in italiano evidenziando i benefici del prodotto e offri una demo."
    ),
    "altro": (
        "Sei un assistente virtuale cortese. "
        "Rispondi in italiano in modo generico e indirizza verso il canale corretto."
    ),
}

def classifica_email(email: str) -> str:
    """Usa l'LLM per classificare l'email in una delle categorie predefinite."""
    categorie = ", ".join(HANDLERS.keys())
    risposta = llm.invoke([
        SystemMessage(content=(
            f"Sei un classificatore. Analizza il testo e rispondi SOLO con una "
            f"di queste parole per taggare il testo di una email (nessun'altra parola): {categorie}"
        )),
        HumanMessage(content=f"Classifica questa email:\n\n{email}"),
    ])
    # Normalizza: prendi solo la prima parola in minuscolo
    categoria = risposta.content.strip().lower().split()[0]
    return categoria if categoria in HANDLERS else "altro"

def gestisci_email(email: str) -> str:
    """Workflow completo: classifica → smista → rispondi."""
    categoria = classifica_email(email)
    print(f"  🔀 Categoria rilevata: [{categoria.upper()}]")
    
    system_prompt = HANDLERS[categoria]
    risposta = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=f"Email del cliente:\n\n{email}\n\nScrivi una risposta."),
    ])
    return risposta.content

# Test con 3 email diverse
emails = [
    "Buongiorno, ho un problema tecnico nel mio account. Non riesco ad accedere al mio account: inserisco la password ma ricevo sempre 'errore 403'. Ho già svuotato la cache del browser.",
    "Ho acquistato il vostro prodotto 10 giorni fa ma non funziona come promesso. Vorrei un rimborso completo.",
    "Siete interessati a una partnership commerciale? Gestiamo una rete di 500 rivenditori in Italia.",
]

for i, email in enumerate(emails, 1):
    print(f"\n{'='*60}")
    print(f"📧 EMAIL #{i}: {email[:70]}...")
    risposta = gestisci_email(email)
    stampa(f"RISPOSTA GENERATA #{i}", risposta)


📧 EMAIL #1: Buongiorno, ho un problema tecnico nel mio account. Non riesco ad acce...
  🔀 Categoria rilevata: [TECNICO]

────────────────────────────────────────────────────────────
📌 RISPOSTA GENERATA #1
────────────────────────────────────────────────────────────
Gentile Cliente,  Grazie per averci contattato. Ho preso visione del
problema relativo all'errore HTTP **403 Forbidden**. Questo codice di
stato indica che il server ha rifiuto la richiesta di accesso a causa
di problemi di autorizzazione o gestione della sessione, e non è
solitamente risolvibile semplicemente svuotando la cache temporanea
(cache) del browser.  Per isolare la causa e ripristinare l'accesso,
ti preghiamo di seguire i seguenti passi tecnici:  1.  **Verifica
tramite Navigazione in Incognito**     Apri una finestra di
navigazione anonima (o "In incognito") nel tuo browser e tenta il
login. Questo esclusione cookies corrotti o dati di sessione locale
(`LocalStorage`) che potrebbero inviare token non validi al se

---
## 4️⃣ Pattern: Orchestrator-Subagents

### Cos'è?
Un **orchestratore** LLM decompone il task in sotto-task e delega a **worker specializzati**,
poi integra i risultati:

```
                    ┌─► [Worker: Ricerca] ─────┐
Input → [Orchestratore] ─► [Worker: Scrittura] ──┼─► [Orchestratore] → Output
                    └─► [Worker: Formattazione] ─┘
```

### Quando usarlo?
Task complessi che richiedono **competenze diverse** e un coordinamento intelligente.
Ogni worker è ottimizzato per un compito specifico.

### Esempio
Produzione di un report di ricerca su un tema: l'orchestratore coordina un ricercatore,
un analista e un redattore.

In [7]:
# ── 4. ORCHESTRATOR-SUBAGENTS ─────────────────────────────────────────────────
# Obiettivo: produrre un report strutturato coordinando 3 worker specializzati.

# Worker 1 — Ricercatore: raccoglie fatti e dati chiave
def worker_ricercatore(argomento: str) -> str:
    print("  🔬 Worker Ricercatore in esecuzione...")
    risposta = llm.invoke([
        SystemMessage(content=(
            "Sei un ricercatore accademico. Rispondi SOLO in italiano. "
            "Elenca 5 fatti chiave, dati o tendenze recenti sull'argomento. "
            "Usa elenco puntato."
        )),
        HumanMessage(content=f"Argomento: {argomento}"),
    ])
    return risposta.content

# Worker 2 — Analista: interpreta i dati e identifica implicazioni
def worker_analista(fatti: str, argomento: str) -> str:
    print("  📊 Worker Analista in esecuzione...")
    risposta = llm.invoke([
        SystemMessage(content=(
            "Sei un analista strategico. Rispondi SOLO in italiano. "
            "Interpreta i dati forniti e identifica 3 implicazioni strategiche importanti."
        )),
        HumanMessage(content=f"Argomento: {argomento}\n\nFatti raccolti:\n{fatti}"),
    ])
    return risposta.content

# Worker 3 — Redattore: produce il report finale
def worker_redattore(fatti: str, analisi: str, argomento: str) -> str:
    print("  ✍️  Worker Redattore in esecuzione...")
    risposta = llm.invoke([
        SystemMessage(content=(
            "Sei un redattore professionale. Rispondi SOLO in italiano. "
            "Produci un report esecutivo conciso (max 300 parole) con: "
            "Titolo, Executive Summary, Fatti Chiave, Analisi, Conclusione."
        )),
        HumanMessage(content=(
            f"Argomento: {argomento}\n\n"
            f"FATTI:\n{fatti}\n\n"
            f"ANALISI:\n{analisi}"
        )),
    ])
    return risposta.content

# Orchestratore — coordina i worker in sequenza
def orchestratore_report(argomento: str) -> str:
    print(f"\n🎯 Orchestratore avviato per: '{argomento}'")
    print("  📋 Fase 1/3: Raccolta fatti...")
    fatti    = worker_ricercatore(argomento)
    
    print("  📋 Fase 2/3: Analisi...")
    analisi  = worker_analista(fatti, argomento)
    
    print("  📋 Fase 3/3: Redazione report...")
    report   = worker_redattore(fatti, analisi, argomento)
    
    return fatti, analisi, report

# Esecuzione
argomento = "impatto dell'AI generativa sul mercato del lavoro"
fatti, analisi, report = orchestratore_report(argomento)

stampa("OUTPUT Worker Ricercatore", fatti)
stampa("OUTPUT Worker Analista", analisi)
stampa("REPORT FINALE (Worker Redattore)", report)


🎯 Orchestratore avviato per: 'impatto dell'AI generativa sul mercato del lavoro'
  📋 Fase 1/3: Raccolta fatti...
  🔬 Worker Ricercatore in esecuzione...
  📋 Fase 2/3: Analisi...
  📊 Worker Analista in esecuzione...
  📋 Fase 3/3: Redazione report...
  ✍️  Worker Redattore in esecuzione...

────────────────────────────────────────────────────────────
📌 OUTPUT Worker Ricercatore
────────────────────────────────────────────────────────────
*   **Stime sulla produttività macroeconomica:** Secondo recenti
analisi di istituzioni quali McKinsey e Goldman Sachs (2023-2024),
l'adozione diffusa dell'intelligenza artificiale generativa potrebbe
aumentare la produttività globale del lavoro fino al 40% nei prossimi
dieci anni, suggerendo che l'impatto netto sul prodotto interno lordo
sia prevalentemente positivo piuttosto che legato alla semplice
riduzione della forza lavoro. *   **Automazione dei compiti rispetto
ai ruoli:** Studi empirici indicano che l'impatto maggiore riguarda
l'automazione di 

---
## 5️⃣ Pattern: Evaluator-Optimizer

### Cos'è?
Un **generatore** produce un output, un **valutatore** lo giudica con un punteggio e feedback,
e il ciclo si ripete finché la qualità non è soddisfacente:

```
Input → [Generatore] → output → [Valutatore] → score ≥ soglia? → Output
               ↑                                    |
               └──────────── feedback ──────────────┘  (se score < soglia)
```

### Quando usarlo?
Quando la qualità dell'output è **misurabile** e si vuole ottenere un risultato
progressivamente migliore senza intervento umano.

### Esempio
Ottimizzazione iterativa di un oggetto per una email di marketing.

In [ ]:
# ── 5. EVALUATOR-OPTIMIZER ────────────────────────────────────────────────────
# Obiettivo: generare e ottimizzare iterativamente l'oggetto di un'email
#            fino a raggiungere un punteggio di qualità ≥ 8/10.

import json
import re

SOGLIA_QUALITA = 8   # Punteggio minimo accettabile (scala 1-10)
MAX_ITERAZIONI = 4   # Numero massimo di tentativi

def genera_oggetto(prodotto: str, feedback: str = "") -> str:
    """Genera (o migliora) l'oggetto di un'email marketing."""
    istruzioni_feedback = (
        f"\n\nFeedback dalla valutazione precedente (incorporalo nella nuova versione):\n{feedback}"
        if feedback else ""
    )
    risposta = llm.invoke([
        SystemMessage(content=(
            "Sei un copywriter specializzato in email marketing. Rispondi in italiano. "
            "Scrivi SOLO l'oggetto dell'email, senza virgolette, senza spiegazioni."
        )),
        HumanMessage(content=(
            f"Scrivi un oggetto accattivante per un'email promozionale su: {prodotto}"
            f"{istruzioni_feedback}"
        )),
    ])
    return risposta.content.strip()

def valuta_oggetto(oggetto: str, prodotto: str) -> dict:
    """Valuta l'oggetto e restituisce {punteggio, feedback}."""
    risposta = llm.invoke([
        SystemMessage(content=(
            "Sei un esperto di email marketing. Valuta l'oggetto di un'email su scala 1-10. "
            "Rispondi SOLO con un oggetto JSON valido nel formato: "
            '{"punteggio": <numero intero>, "feedback": "<stringa in italiano>"}'
            " — nessun altro testo."
        )),
        HumanMessage(content=(
            f"Prodotto: {prodotto}\n"
            f"Oggetto da valutare: {oggetto}\n\n"
            "Valuta considerando: chiarezza, urgenza, personalizzazione, lunghezza ottimale (max 50 chars), originalità."
        )),
    ])
    # Estrai il JSON dalla risposta (gestisce eventuali testo extra)
    testo = risposta.content.strip()
    match = re.search(r'\{[^}]+\}', testo, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except json.JSONDecodeError:
            pass
    # Fallback se il parsing fallisce
    return {"punteggio": 5, "feedback": testo}

# Loop Evaluator-Optimizer
prodotto = "corso online di machine learning per principianti"
feedback = ""
storia   = []   # Tiene traccia di ogni iterazione

print(f"\n🎯 Obiettivo: oggetto email per '{prodotto}'")
print(f"   Soglia qualità: {SOGLIA_QUALITA}/10 | Max iterazioni: {MAX_ITERAZIONI}\n")

for i in range(1, MAX_ITERAZIONI + 1):
    print(f"{'─'*50}")
    print(f"🔄 Iterazione {i}/{MAX_ITERAZIONI}")
    
    # Genera
    oggetto = genera_oggetto(prodotto, feedback)
    print(f"  ✏️  Generato: '{oggetto}'")
    
    # Valuta
    valutazione = valuta_oggetto(oggetto, prodotto)
    punteggio   = valutazione.get("punteggio", 0)
    feedback    = valutazione.get("feedback", "")
    
    print(f"  ⭐ Punteggio: {punteggio}/10")
    print(f"  💬 Feedback: {feedback}")
    
    storia.append({"iterazione": i, "oggetto": oggetto, "punteggio": punteggio})
    
    if punteggio >= SOGLIA_QUALITA:
        print(f"\n✅ Soglia raggiunta all'iterazione {i}!")
        break
    elif i < MAX_ITERAZIONI:
        print(f"  ↩️  Punteggio sotto soglia, rigenero con il feedback...")
else:
    print(f"\n⚠️  Numero massimo di iterazioni raggiunto.")

# Riepilogo
print(f"\n{'='*50}")
print("📊 RIEPILOGO ITERAZIONI:")
for item in storia:
    marker = "✅" if item["punteggio"] >= SOGLIA_QUALITA else "❌"
    print(f"  {marker} [{item['iterazione']}] {item['punteggio']}/10 — {item['oggetto']}")
print(f"\n🏆 OGGETTO FINALE: {storia[-1]['oggetto']}")


🎯 Obiettivo: oggetto email per 'corso online di machine learning per principianti'
   Soglia qualità: 8/10 | Max iterazioni: 4

──────────────────────────────────────────────────
🔄 Iterazione 1/4
  ✏️  Generato: 'Machine Learning da zero: Inizia oggi senza esperienza pregressa'
  ⭐ Punteggio: 6/10
  💬 Feedback: Oggetto chiaro e diretto ma troppo lungo (64 caratteri). Viola il limite di 50 caratteri richiesto per ottimizzare l'apertura su mobile. Suggerimento: riduci a circa 40-45 caratteri concentrandoti sui benefici chiave.
  ↩️  Punteggio sotto soglia, rigenero con il feedback...
──────────────────────────────────────────────────
🔄 Iterazione 2/4
  ✏️  Generato: ''
  ⭐ Punteggio: 1/10
  💬 Feedback: Oggetto mancante nel prompt fornito. Non è possibile valutare i criteri di chiarezza, urgenza, personalizzazione, lunghezza e originalità in assenza del testo dell'oggetto.
  ↩️  Punteggio sotto soglia, rigenero con il feedback...
──────────────────────────────────────────────────
🔄 Itera

KeyboardInterrupt: 

: 

---
## 🧭 Riepilogo: quando usare ogni pattern?

| Pattern | Struttura | Caso d'uso tipico |
|---------|-----------|-------------------|
| **Prompt Chaining** | Lineare | Pipeline con fasi ordinate e dipendenti |
| **Parallelization** | Fan-out / Fan-in | Task indipendenti, voting, batch processing |
| **Routing** | Classificazione + dispatch | Input eterogenei con handler specializzati |
| **Orchestrator-Subagents** | Coordinamento gerarchico | Task complessi con competenze diverse |
| **Evaluator-Optimizer** | Loop con feedback | Ottimizzazione iterativa della qualità |

---

### 🔑 Workflow vs Agenti — differenza fondamentale

| | **Workflow** | **Agente** |
|---|---|---|
| **Controllo del flusso** | Predefinito dallo sviluppatore | Deciso autonomamente dall'LLM |
| **Determinismo** | ✅ Deterministico | ❌ Non deterministico |
| **Strumenti/Tools** | Chiamate fisse e note a priori | Selezionati dinamicamente dall'LLM |
| **Adattabilità** | Bassa (flusso rigido) | Alta (può cambiare piano) |
| **Affidabilità** | Alta | Dipende dall'LLM |
| **Quando preferirlo** | Task ben definiti e ripetibili | Task aperti e imprevedibili |

> 💡 **Regola pratica:** inizia sempre con un workflow. Passa agli agenti solo quando
> il flusso non può essere predefinito a causa della complessità o variabilità del task.